<a href="https://colab.research.google.com/github/Railyng/Clasificador-de-Sentimientos---BellezaNatural/blob/main/ClasificadorDeSentimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificador de Sentimientos - BellezaNatural**

Este proyecto tiene como objetivo desarrollar un sistema capaz de analizar comentarios de clientes y clasificarlos como positivos, negativos o neutros utilizando técnicas de Procesamiento de Lenguaje Natural (NLP) y Redes Neuronales.

In [11]:
import pandas as pd  # Para manejar datos en forma de tabla (DataFrame)
import numpy as np   # Para operaciones matemáticas y manejo de arreglos
import re            # Para limpiar texto usando expresiones regulares

import nltk          # Librería para procesamiento de lenguaje natural

# Herramientas de Machine Learning
from sklearn.model_selection import train_test_split  # Para dividir datos
from sklearn.preprocessing import LabelEncoder        # Para convertir texto a números
from sklearn.metrics import classification_report     # Para evaluar el modelo

# Librerías de Deep Learning
from tensorflow.keras.models import Sequential        # Para crear la red neuronal
from tensorflow.keras.layers import Dense, Embedding, Flatten  # Capas de la red
from tensorflow.keras.preprocessing.text import Tokenizer       # Convertir texto a números
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Igualar tamaños de texto

In [12]:
# Creamos un diccionario con comentarios y su sentimiento
data = {
    "comentario": [
        "El producto es excelente",
        "Muy mala calidad",
        "Está bien, pero puede mejorar",
        "Me encantó, lo recomiendo",
        "No me gustó para nada",
        "Es normal, nada especial",
        "Llegó rápido y en buen estado",
        "Se dañó en pocos días",
        "Cumple su función",
        "Pésimo servicio"
    ],
    "sentimiento": [
        "positivo", "negativo", "neutro",
        "positivo", "negativo", "neutro",
        "positivo", "negativo", "neutro", "negativo"
    ]
}

# Convertimos el diccionario en un DataFrame (tabla)
df = pd.DataFrame(data)

# Mostramos los primeros datos
df.head()

,comentario,sentimiento
0,El producto es excelente,positivo
1,Muy mala calidad,negativo
2,"Está bien, pero puede mejorar",neutro
3,"Me encantó, lo recomiendo",positivo
4,No me gustó para nada,negativo


In [13]:
# Función para limpiar texto
def limpiar_texto(texto):
    texto = texto.lower()  # Convertir todo a minúsculas
    texto = re.sub(r'[^a-záéíóúñ ]', '', texto)  # Eliminar símbolos y números
    return texto

# Aplicamos la función a todos los comentarios
df["comentario"] = df["comentario"].apply(limpiar_texto)

# Verificamos los cambios
df.head()

,comentario,sentimiento
0,el producto es excelente,positivo
1,muy mala calidad,negativo
2,está bien pero puede mejorar,neutro
3,me encantó lo recomiendo,positivo
4,no me gustó para nada,negativo


In [14]:
# Creamos el tokenizador (máximo 1000 palabras)
tokenizer = Tokenizer(num_words=1000)

# Aprende el vocabulario del texto
tokenizer.fit_on_texts(df["comentario"])

# Convierte cada comentario en números
secuencias = tokenizer.texts_to_sequences(df["comentario"])

# Igualamos la longitud de todos los textos a 10 palabras
X = pad_sequences(secuencias, maxlen=10)

# Mostramos el resultado
X

array([[ 0,  0,  0,  0,  0,  0,  5,  6,  1,  7],
       [ 0,  0,  0,  0,  0,  0,  0,  8,  9, 10],
       [ 0,  0,  0,  0,  0, 11, 12, 13, 14, 15],
       [ 0,  0,  0,  0,  0,  0,  2, 16, 17, 18],
       [ 0,  0,  0,  0,  0, 19,  2, 20, 21,  3],
       [ 0,  0,  0,  0,  0,  0,  1, 22,  3, 23],
       [ 0,  0,  0,  0, 24, 25, 26,  4, 27, 28],
       [ 0,  0,  0,  0,  0, 29, 30,  4, 31, 32],
       [ 0,  0,  0,  0,  0,  0,  0, 33, 34, 35],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 36, 37]], dtype=int32)

In [15]:
# Creamos el codificador
encoder = LabelEncoder()

# Convertimos etiquetas de texto a números
y = encoder.fit_transform(df["sentimiento"])

# Mostramos las etiquetas convertidas
y

array([2, 0, 1, 2, 0, 1, 2, 0, 1, 0])

In [16]:
# Dividimos los datos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Mostramos cantidad de datos
print("Datos de entrenamiento:", len(X_train))
print("Datos de prueba:", len(X_test))

Datos de entrenamiento: 8
Datos de prueba: 2


In [17]:
# Creamos el modelo secuencial
model = Sequential()

# Capa Embedding: convierte palabras en vectores numéricos
model.add(Embedding(input_dim=1000, output_dim=16, input_length=10))

# Aplanamos los datos para pasarlos a la red
model.add(Flatten())

# Capa oculta con función de activación ReLU
model.add(Dense(16, activation='relu'))

# Capa de salida con 3 neuronas (positivo, negativo, neutro)
model.add(Dense(3, activation='softmax'))

# Compilamos el modelo
model.compile(
    loss='sparse_categorical_crossentropy',  # Función de error
    optimizer='adam',                        # Algoritmo de optimización
    metrics=['accuracy']                    # Métrica de evaluación
)

# Mostramos resumen del modelo
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Entrenamos el modelo con los datos
history = model.fit(
    X_train,          # Datos de entrada
    y_train,          # Etiquetas
    epochs=10,        # Número de iteraciones
    validation_data=(X_test, y_test)  # Validación
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3750 - loss: 1.0974 - val_accuracy: 0.0000e+00 - val_loss: 1.0925
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.3750 - loss: 1.0904 - val_accuracy: 0.0000e+00 - val_loss: 1.0909
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.5000 - loss: 1.0838 - val_accuracy: 0.0000e+00 - val_loss: 1.0891
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.6250 - loss: 1.0776 - val_accuracy: 0.5000 - val_loss: 1.0879
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.7500 - loss: 1.0720 - val_accuracy: 0.5000 - val_loss: 1.0874
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.7500 - loss: 1.0668 - val_accuracy: 0.5000 - val_loss: 1.0869
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.6250 - loss: 1.0613 - val_accuracy: 0.5000 - val_loss: 1.0864
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.6250 - loss: 1.0561 - val_accuracy: 0.5000 - 

In [19]:
# Predicciones del modelo
y_pred = model.predict(X_test)

# Convertimos probabilidades a clases
y_pred_classes = np.argmax(y_pred, axis=1)

# Mostramos métricas
print(classification_report(y_test, y_pred_classes))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
# Función para predecir sentimiento
def predecir_sentimiento(texto):
    texto = limpiar_texto(texto)  # Limpiar texto
    seq = tokenizer.texts_to_sequences([texto])  # Convertir a números
    padded = pad_sequences(seq, maxlen=10)  # Ajustar tamaño

    pred = model.predict(padded)  # Predicción
    clase = np.argmax(pred)       # Obtener clase

    # Convertir número a etiqueta
    return encoder.inverse_transform([clase])[0]

# Pruebas con nuevos comentarios
print(predecir_sentimiento("Me encantó este producto"))
print(predecir_sentimiento("Muy malo"))
print(predecir_sentimiento("Está regular"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
negativo
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
negativo
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
negativo


## **Conclusión**

El sistema desarrollado permite analizar automáticamente comentarios de clientes, clasificándolos en positivos, negativos o neutros.

Se aplicaron técnicas de procesamiento de lenguaje natural y redes neuronales, logrando automatizar el análisis de opiniones.

Este tipo de sistema ayuda a las empresas a mejorar la toma de decisiones y comprender mejor a sus clientes.